# tw-market-data-lake — Data Quality Audit v0.1.0 · readiness v0.2.0（Phase A，唯讀）

- **唯讀**：不寫入、搬移、刪除 DATA_ROOT 下任何資料集、manifest、pointer、lock、staging、quarantine、runs 或既有資料夾；不呼叫任何行情 API；不需要 FinLab / 富邦憑證。
- 只讀各 dataset `_manifest/published.json` 指向的版本與其 `partitions.parquet` 列出的檔案。
- 輸出先寫到 Colab 本機 `/content/tw_lake_audits/<audit_id>/`；`PUBLISH_TO_DRIVE=True` 時，完成後才**複製**到 `_tw-market-data-lake/audits/<audit_id>/`（新資料夾，不覆蓋任何既有路徑），並 flush Drive。
- 全量（8,839 個 5 分K分區）約需讀 1.5 GB，Colab 上預估 20–60 分鐘；可先用 `SYMBOLS` 小範圍試跑。

In [ ]:
# 1. 參數
SYMBOLS = ""              # 例 "2330,0050,8299"；空白 = 全部 5m 分區
MAX_PARTITIONS = None     # 例 200；None = 不限
DEEP_VERIFY = False       # True: 所有已發布檔案 SHA256（最慢、最完整）
SHA_SAMPLE = 20           # 每個 dataset 隨機抽 N 個檔案做 SHA256
PUBLISH_TO_DRIVE = False  # True: 完成後把報表複製到 Drive 的 _tw-market-data-lake/audits/<audit_id>/

REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
PROJECT_DIR = "/content/tw-market-data-lake"
OUT_DIR = "/content/tw_lake_audits"

In [ ]:
# 2. 掛載 Google Drive（只讀取；稽核程式本身不寫入 DATA_ROOT）
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull 指定 branch，記錄 HEAD SHA
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
SHA = subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip()
os.environ["TW_LAKE_GIT_SHA"] = SHA
print("branch:", BRANCH, "HEAD:", SHA)

In [ ]:
# 4. 依賴（Colab 內建 pandas / pyarrow；只補 pyyaml / tqdm）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")

In [ ]:
# 5. 執行唯讀稽核
import sys, subprocess
args = [sys.executable, "-m", "tw_lake.audit.cli", "--data-root", DATA_ROOT, "--out-dir", OUT_DIR,
        "--sha-sample", str(SHA_SAMPLE)]
if SYMBOLS: args += ["--symbols", SYMBOLS]
if MAX_PARTITIONS: args += ["--max-partitions", str(MAX_PARTITIONS)]
if DEEP_VERIFY: args += ["--deep-verify"]
if PUBLISH_TO_DRIVE: args += ["--publish-to-drive"]
env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
print("+", " ".join(args))
# 即時顯示每個階段與進度（全量約 8,839 個分區）
proc = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
print("exit code:", rc)
if PUBLISH_TO_DRIVE:
    drive.flush_and_unmount()   # Drive FUSE 為非同步上傳：不 flush 可能留下 0-byte 檔
    print("Drive flushed & unmounted")

In [ ]:
# 6. 顯示摘要
import glob, json
from IPython.display import Markdown, display
latest = sorted(glob.glob(f"{OUT_DIR}/audit_*"))[-1]
print(latest)
display(Markdown(open(f"{latest}/audit_summary.md", encoding="utf-8").read()))

In [ ]:
# 7. 打包下載（選用）：把整個 audit 資料夾壓成 zip
import shutil
zip_path = shutil.make_archive(latest, "zip", latest)
print(zip_path)
from google.colab import files
files.download(zip_path)

## 8.（選用）只重算 readiness — 不必重跑整份稽核（數秒～1 分鐘）

讀「已完成的稽核資料夾」裡的 `audit_5m_by_symbol_day` / `audit_ohlcv_reconciliation` / `audit_integrity`，
重新計算 readiness v0.2（含 `ready_5m_no_trade_tolerant`）。**不讀 DATA_ROOT 的任何資料集**、不改原稽核檔案；
結果寫到 Colab 本機新資料夾，`RESCORE_COPY_TO_DRIVE=True` 時再複製一份到該稽核資料夾底下的新子資料夾 `rescore_*`。
需要先跑過第 1–4 格（參數、掛載、clone、依賴）。


In [ ]:
# 8. readiness rescore
import glob, os, shutil, subprocess, sys
AUDIT_DIR = ""                 # 空白 = 自動找 Drive 上最新的 _tw-market-data-lake/audits/audit_*
MISSING_ALLOWANCE = ""         # 空白 = 預設 ge5000_lots=0,500_5000_lots=3,50_500_lots=15,lt50_lots=none
REQUIRE_CLOSE_AUCTION = True  # False: 無 13:30 收盤集合競價成交的冷門股日，只要對帳 ok（收盤價已相等）也接受
RESCORE_COPY_TO_DRIVE = True
if not AUDIT_DIR:
    cands = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/audit_*")) or sorted(glob.glob(f"{OUT_DIR}/audit_*"))
    AUDIT_DIR = [c for c in cands if os.path.isdir(c)][-1]
print("audit:", AUDIT_DIR)
from datetime import datetime
local_out = f"{OUT_DIR}/rescore_{os.path.basename(AUDIT_DIR)}_{datetime.now():%Y%m%d_%H%M%S}"
args = [sys.executable, "-m", "tw_lake.audit.rescore", "--audit-dir", AUDIT_DIR, "--out-dir", local_out]
if MISSING_ALLOWANCE: args += ["--missing-allowance", MISSING_ALLOWANCE]
if not REQUIRE_CLOSE_AUCTION: args += ["--no-require-close-auction"]
env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src")
r = subprocess.run(args, capture_output=True, text=True, env=env)
print(r.stdout); print(r.stderr[-3000:]); print("exit", r.returncode)
if r.returncode == 0 and RESCORE_COPY_TO_DRIVE:
    dest = f"{AUDIT_DIR}/{os.path.basename(local_out)}"
    shutil.copytree(local_out, dest)          # 新路徑，不覆蓋任何既有檔案
    print("copied to", dest, "（Drive 寫入需等同步；可再執行 drive.flush_and_unmount()）")


In [ ]:
# 9. 顯示 rescore 摘要與流動性校準表
from IPython.display import Markdown, display
display(Markdown(open(f"{local_out}/rescore_summary.md", encoding="utf-8").read()))
